# 05 - Treino Final e Exporta??o

Reajuste do melhor modelo e salvamento em `models/`.

In [ ]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

data = pd.read_parquet("../data/processed/flights_sample.parquet")
features = ["AIRLINE","ORIGIN_AIRPORT","DESTINATION_AIRPORT","MONTH","DAY_OF_WEEK","DEP_HOUR","DISTANCE","IS_WEEKEND"]
X = data[features]
y = data["DELAYED"]

cat_features = ["AIRLINE","ORIGIN_AIRPORT","DESTINATION_AIRPORT"]
num_features = ["MONTH","DAY_OF_WEEK","DEP_HOUR","DISTANCE","IS_WEEKEND"]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_features)
])

rf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=150, max_depth=18, random_state=42, n_jobs=-1
    ))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
rf.fit(X_train, y_train)
auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
print(f"ROC-AUC holdout: {auc:.3f}")

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)
out_path = models_dir / "random_forest_delay_model.pkl"
joblib.dump(rf, out_path)
print(f"Modelo salvo em {out_path}")